<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/08_3_%EB%8C%80%ED%99%94%EC%83%81%ED%83%9C_%EA%B8%B0%EB%B0%98_%EC%9D%91%EB%8B%B5%EC%A0%9C%EC%96%B4_%EA%B5%AC%ED%98%84_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 08-3: 대화상태 기반 응답제어 구현  

### 실습목표

- 대화의 진행 단계를 **상태(State)** 로 정의하고, 각 상태에 따른 응답 로직을 설계할 수 있다.  

- 프롬프트에 현재 **상태 정보를 주입(State Injection)** 하여 LLM이 맥락에 맞는 답변을 생성하도록 제어할 수 있다.  

- 사용자의 입력에 따라 **상태를 전이(Transition)** 시키며 목표 지향적인 대화 시나리오를 완수하는 법을 습득한다.  

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [1]:
# 기존 설치를 무시하고 최신 버전으로 강제 재설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

### 2. Gemini LLM 로드

Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

In [3]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


3. 상태 기반 제어의 구조  

- AI 에이전트는 단순히 질문에 답하는 것이 아니라, 자신이 현재 어떤 '단계'에 있는지 인지하고 그에 맞는 행동을 합니다.  

    - Inquiry (문의 단계): 사용자의 질문에 답하며 정보를 수집함  
    - Confirming (확인 단계): 수집된 정보를 바탕으로 최종 의사를 확인하며, 이때는 다른 주제로 새는 것을 방지함  
    - Completed (완료 단계): 모든 프로세스가 종료된 상태

- 기존 08-2의 로직에 대화 상태(State) 변수와 상태별 프롬프트 제어 기능을 추가한 최종 보완 코드입니다.  

In [4]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_community.chat_message_histories import ChatMessageHistory

# --- 1. 대화 상태 정의 (Dialog State) ---
history_db = ChatMessageHistory()
# 현재 대화의 상태(status)와 슬롯을 통합 관리합니다.
dialog_state = {
    "status": "INQUIRY", # 초기 상태: 정보 문의 및 수집
    "slots": {"name": None, "location": None, "topic": None}
}

# --- 2. LLM 및 상태 기반 프롬프트 설정 ---
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

# 프롬프트에 {current_status}를 추가하여 LLM에게 현재 단계를 인지시킵니다.
template = """너는 학습 상담 에이전트야. 현재 너의 상태는 [{current_status}]야.

[상태별 지침]
- INQUIRY: 친절하게 질문에 답하며 이름, 지역, 주제 정보를 수집해.
- CONFIRMING: 수집된 정보를 확인하고 최종 동의를 얻어. 다른 질문은 정중히 거절하고 확인에 집중해.
- COMPLETED: 상담이 끝났음을 알리고 행운을 빌어줘.

현재 파악된 정보: 이름({name}), 지역({location}), 주제({topic})
사용자: {question}
답변:"""

prompt = PromptTemplate.from_template(template)

# --- 3. 데이터 흐름 구성 ---
lcel_chain = (
    {
        "question": RunnablePassthrough(),
        "history": RunnableLambda(lambda _: "\n".join([f"{m.type}: {m.content}" for m in history_db.messages])),
        "current_status": lambda _: dialog_state["status"], # 현재 상태 주입
        "name": lambda _: dialog_state["slots"]["name"] or "미파악",
        "location": lambda _: dialog_state["slots"]["location"] or "미파악",
        "topic": lambda _: dialog_state["slots"]["topic"] or "미파악"
    }
    | prompt | llm | StrOutputParser()
)

# --- 4. 상태 전이(State Transition) 로직 ---
def chat(user_input):
    # [NLU 단계] 슬롯 업데이트 (08-1 개념)
    if "제미니" in user_input: dialog_state["slots"]["name"] = "제미니"
    if "서울" in user_input: dialog_state["slots"]["location"] = "서울"
    if "랭체인" in user_input: dialog_state["slots"]["topic"] = "랭체인"

    # [상태 전이 결정] 모든 슬롯이 채워지면 CONFIRMING으로 변경
    if all(dialog_state["slots"].values()) and dialog_state["status"] == "INQUIRY":
        dialog_state["status"] = "CONFIRMING"
        print("[*] 시스템: 상태가 'CONFIRMING'으로 전이되었습니다.")

    # 사용자가 동의하면 COMPLETED로 변경
    if dialog_state["status"] == "CONFIRMING" and ("응" in user_input or "확인" in user_input):
        dialog_state["status"] = "COMPLETED"
        print("[*] 시스템: 상태가 'COMPLETED'로 전이되었습니다.")

    response = lcel_chain.invoke(user_input)

    history_db.add_user_message(user_input)
    history_db.add_ai_message(response)
    return response

print("대화 상태 기반 응답 제어 에이전트 로드 완료!")

대화 상태 기반 응답 제어 에이전트 로드 완료!


3. 실습 테스트 및 검증  

- AI 에이전트가 상태(INQUIRY → CONFIRMING)에 따라 답변의 '목적'을 어떻게 바꾸는지 확인합니다.  

In [5]:
# --- [08-3 테스트 시나리오] ---

# 단계 1: 정보 수집 (INQUIRY 상태)
print(f"A: {chat('안녕, 내 이름은 제미니야. 서울에서 랭체인을 배우고 있어.')}\n")

# 단계 2: 상태 전이 확인 (모든 슬롯이 채워져 CONFIRMING 상태가 됨)
# 이때 사용자가 딴소리를 해도 에이전트는 확인을 유도해야 함
print(f"Q: 다른 거 물어봐도 돼?\nA: {chat('다른 거 물어봐도 돼?')}\n")

# 단계 3: 프로세스 완료 (COMPLETED 상태)
print(f"Q: 응, 정보가 다 맞아. 확인!\nA: {chat('응, 정보가 다 맞아. 확인!')}\n")

[*] 시스템: 상태가 'CONFIRMING'으로 전이되었습니다.
A: 제미니님, 현재까지 수집된 정보를 확인하겠습니다.

*   **이름:** 제미니
*   **지역:** 서울
*   **주제:** 랭체인

이 정보가 정확한가요? 이대로 학습 상담을 진행해도 될까요?

Q: 다른 거 물어봐도 돼?
A: 제미니님, 죄송하지만 지금은 수집된 정보를 최종 확인하는 단계입니다. 다른 질문은 정중히 거절하고 확인에 집중해야 합니다.

먼저, 수집된 정보가 맞는지 확인해 주시겠어요?

**제미니님께서 서울 지역에서 랭체인 관련 학습 상담을 진행하는 것이 맞을까요?**

[*] 시스템: 상태가 'COMPLETED'로 전이되었습니다.


ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 45.872588772s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr

#### 보강 포인트

- **상태 주입(State Injection)**: LLM에게 현재 CONFIRMING 단계임을 알려줌으로써, 모델이 스스로 대화의 주도권을 잡고 확인 절차에 집중하도록 만들었습니다.  

- **동적 시나리오**: 단순한 Q&A를 넘어, 조건(모든 슬롯 충족)이 맞을 때만 다음 단계로 넘어가는 비즈니스 프로세스를 코드로 구현했습니다.  

- **대화 제어력**: 사용자가 중간에 질문을 던져도 현재 상태의 목적(확인)을 잊지 않고 복귀하는 '목표 지향적' 에이전트의 기초를 완성했습니다.  